### IMPLEMENTING MULTI-HEAD ATTENTION WITH WEIGHT SPLITS

Instead of maintaining 2 separate classes, MultiHeadAttentionWrapper and CausalAttention, we can combine both of these concepts into single MultiHeadAttention class

Also we will modify the implemntation to make it more efficient

In the MultiHeadAttentionWrapper, multiple heads are implemented by creating a list of CausalAttention objects (self.heads) each represneting a separate head

The CausalAttentionClass independently perform the attention mechanism and the results from each are computed.

In contrast, the following MultiHeadAttentionclass integrates the muti head functionality in a single class

It splits the input nto multiple heads by reshaping the projected Query, Key and Value tensors nd the combines the results from these heads after computing attention.

In [1]:
import torch
import torch.nn as nn

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super.__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)



    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # (b, num_tokens, d_out)
        queries = self.W_query(x)  # (b, num_tokens, d_out)
        values = self.W_value(x)  # (b, num_tokens, d_out)

        # we implicitly split the matrix by adding a new dimension for the heads "num_heads"
        # unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # transpose to get dimensions: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        # we wan th grouping to be the heads, so we swap the num_heads and num_tokens dimensions not on tokens

        keys = keys.transpose(1, 2)  # (b, num_heads, num_tokens, head_dim)
        queries = queries.transpose(1, 2)  # (b, num_heads, num_tokens, head_dim)
        values = values.transpose(1, 2)  # (b, num_heads, num_tokens, head_dim)

